# Joint Distribution Construction and Synthetic Data Generation

This notebook constructs joint distributions for numerical variables within each categorical combination and generates synthetic datasets with pair plot comparisons.

## Workflow
1. **Load CSV data** with `cat_` and `num_` prefixed columns
2. **Group by categorical combinations** (e.g., cat_region + cat_segment)
3. **Construct joint distributions** for numerical variables in each group
4. **Generate synthetic data** of desired size preserving joint distributions
5. **Create pair plots** comparing original vs synthetic for each category combination

## Expected Data Format
```
cat_region, cat_type, num_sales, num_profit, num_customers
North,     Premium,   150.2,     45.6,      120
South,     Standard,  98.5,      22.3,      85
```

## Step 1: Setup and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import multivariate_normal, gaussian_kde
from sklearn.preprocessing import StandardScaler
import warnings
import itertools
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('default')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
np.random.seed(42)

print("✅ Libraries imported successfully")
print("📊 Ready for joint distribution analysis")

# Global configuration
DESIRED_SYNTHETIC_SIZE = 1000  # Change this to your desired synthetic dataset size
GENERATION_METHOD = 'multivariate_normal'  # Options: 'multivariate_normal', 'kde', 'bootstrap'

print(f"Target synthetic dataset size: {DESIRED_SYNTHETIC_SIZE}")
print(f"Generation method: {GENERATION_METHOD}")

## Step 2: Load Your CSV Data

**Important**: Update the file path below to your CSV file.

Your CSV should have:
- Categorical columns starting with `cat_`
- Numerical columns starting with `num_`

In [ ]:
# CHANGE THIS PATH TO YOUR CSV FILE
CSV_FILE_PATH = 'your_data.csv'  # ← Update this with your file path

def load_data(file_path):
    """Load and validate CSV data"""
    try:
        df = pd.read_csv(file_path)
        print(f"✅ Successfully loaded: {file_path}")
        print(f"Dataset shape: {df.shape}")
        return df
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        print("Creating sample data for demonstration...")
        return create_sample_data()
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        print("Creating sample data for demonstration...")
        return create_sample_data()

def create_sample_data(n_samples=1200):
    """Create sample data for demonstration"""
    print("🎲 Creating sample business data...")
    
    # Categorical variables
    regions = ['North', 'South', 'East', 'West']
    segments = ['Enterprise', 'SMB', 'Consumer']
    
    cat_region = np.random.choice(regions, n_samples, p=[0.3, 0.25, 0.25, 0.2])
    cat_segment = np.random.choice(segments, n_samples, p=[0.2, 0.5, 0.3])
    
    # Initialize numerical arrays
    num_revenue = np.zeros(n_samples)
    num_profit = np.zeros(n_samples)
    num_customers = np.zeros(n_samples)
    num_satisfaction = np.zeros(n_samples)
    
    # Create different joint distributions for each combination
    for i in range(n_samples):
        region = cat_region[i]
        segment = cat_segment[i]
        
        # Region-based parameters
        region_multipliers = {
            'North': (1.2, 1.1),
            'South': (1.0, 1.0),
            'East': (1.1, 1.05),
            'West': (0.9, 0.95)
        }
        
        # Segment-based parameters
        segment_params = {
            'Enterprise': {'base_rev': 200, 'margin': 0.25, 'cust_base': 150, 'sat_base': 4.2},
            'SMB': {'base_rev': 100, 'margin': 0.15, 'cust_base': 80, 'sat_base': 3.8},
            'Consumer': {'base_rev': 50, 'margin': 0.10, 'cust_base': 200, 'sat_base': 3.5}
        }
        
        rev_mult, other_mult = region_multipliers[region]
        params = segment_params[segment]
        
        # Generate correlated variables
        base_revenue = params['base_rev'] * rev_mult
        base_customers = params['cust_base'] * other_mult
        
        # Create correlation between revenue and customers
        correlation = 0.7
        noise1 = np.random.normal(0, 1)
        noise2 = correlation * noise1 + np.sqrt(1 - correlation**2) * np.random.normal(0, 1)
        
        num_revenue[i] = base_revenue + 25 * noise1
        num_customers[i] = base_customers + 20 * noise2
        num_profit[i] = num_revenue[i] * params['margin'] + np.random.normal(0, 5)
        num_satisfaction[i] = params['sat_base'] + 0.3 * np.random.normal(0, 1)
        
        # Ensure positive values
        num_revenue[i] = max(10, num_revenue[i])
        num_customers[i] = max(5, num_customers[i])
        num_profit[i] = max(0, num_profit[i])
        num_satisfaction[i] = max(1, min(5, num_satisfaction[i]))
    
    df = pd.DataFrame({
        'cat_region': cat_region,
        'cat_segment': cat_segment,
        'num_revenue': num_revenue,
        'num_profit': num_profit,
        'num_customers': num_customers,
        'num_satisfaction': num_satisfaction
    })
    
    print(f"✅ Sample data created: {df.shape}")
    return df

# Load the data
df = load_data(CSV_FILE_PATH)

# Show basic info
print(f"\nDataset Overview:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

## Step 3: Data Validation and Analysis

In [ ]:
# Identify column types
cat_cols = [col for col in df.columns if col.startswith('cat_')]
num_cols = [col for col in df.columns if col.startswith('num_')]

print(f"📊 DATA STRUCTURE ANALYSIS")
print(f"{'='*50}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numerical columns ({len(num_cols)}): {num_cols}")

if not cat_cols:
    print("❌ ERROR: No categorical columns found!")
    print("   Please ensure categorical columns start with 'cat_'")
elif not num_cols:
    print("❌ ERROR: No numerical columns found!")
    print("   Please ensure numerical columns start with 'num_'")
else:
    print("✅ Data structure is valid")

# Analyze categorical combinations
print(f"\n📈 CATEGORICAL COMBINATIONS ANALYSIS")
print(f"{'='*50}")

# Group by all categorical columns
grouped = df.groupby(cat_cols)
group_sizes = grouped.size().sort_values(ascending=False)

print(f"Total unique combinations: {len(group_sizes)}")
print(f"Average group size: {group_sizes.mean():.1f}")
print(f"Largest group: {group_sizes.max()} samples")
print(f"Smallest group: {group_sizes.min()} samples")

print(f"\nGroup sizes by combination:")
for combination, size in group_sizes.items():
    print(f"  {combination}: {size} samples")

# Statistical summary by group
print(f"\n📊 NUMERICAL STATISTICS BY GROUP")
print(f"{'='*50}")
group_stats = df.groupby(cat_cols)[num_cols].agg(['mean', 'std']).round(2)
print(group_stats)

## Step 4: Visualize Original Data

In [ ]:
# Create overall pair plot for original data
print("Creating pair plot for original data...")

# Create combined categorical column for coloring
df_plot = df.copy()
if len(cat_cols) == 1:
    df_plot['cat_combined'] = df_plot[cat_cols[0]]
else:
    df_plot['cat_combined'] = df_plot[cat_cols].apply(
        lambda x: ' + '.join(x.astype(str)), axis=1
    )

# Create pair plot
g = sns.pairplot(
    df_plot[num_cols + ['cat_combined']], 
    hue='cat_combined',
    diag_kind='hist',
    plot_kws={'alpha': 0.7, 's': 25},
    height=2.5
)

g.fig.suptitle('Original Data - Joint Distributions by Category Combinations', 
               y=1.02, fontsize=14, fontweight='bold')

# Improve legend
g._legend.set_title('Category Combinations')
g._legend.set_bbox_to_anchor((1.05, 0.8))

plt.tight_layout()
plt.show()

print("✅ Original data visualization complete")
print(f"Each color represents a different combination of {', '.join(cat_cols)}")

## Step 5: Joint Distribution Construction Framework

In [ ]:
class JointDistributionSynthesizer:
    """
    Constructs joint distributions for each categorical combination
    and generates synthetic data preserving these relationships
    """
    
    def __init__(self, method='multivariate_normal', min_samples=5):
        self.method = method
        self.min_samples = min_samples
        self.distributions = {}
        self.cat_cols = None
        self.num_cols = None
        self.fitted = False
        
    def fit(self, df, cat_cols, num_cols):
        """Fit joint distributions for each categorical combination"""
        
        self.cat_cols = cat_cols
        self.num_cols = num_cols
        
        print(f"🔧 FITTING JOINT DISTRIBUTIONS")
        print(f"{'='*50}")
        print(f"Method: {self.method}")
        print(f"Minimum samples per group: {self.min_samples}")
        
        grouped = df.groupby(cat_cols)
        successful_fits = 0
        
        for combination, group in grouped:
            group_size = len(group)
            
            if group_size < self.min_samples:
                print(f"  ⚠️  Skipping {combination}: only {group_size} samples")
                continue
            
            print(f"  🔄 Fitting {combination}: {group_size} samples")
            
            # Extract numerical data
            num_data = group[num_cols].values
            
            try:
                if self.method == 'multivariate_normal':
                    mean = np.mean(num_data, axis=0)
                    cov = np.cov(num_data.T)
                    
                    # Add small regularization to diagonal if needed
                    if len(num_cols) > 1:
                        cov += np.eye(len(num_cols)) * 1e-6
                    
                    # Test sampling
                    _ = multivariate_normal.rvs(mean=mean, cov=cov, size=2)
                    
                    self.distributions[combination] = {
                        'method': 'multivariate_normal',
                        'mean': mean,
                        'cov': cov,
                        'n_samples': group_size,
                        'original_data': num_data
                    }
                    
                elif self.method == 'kde':
                    if len(num_cols) == 1:
                        kde = gaussian_kde(num_data.T[0])
                    else:
                        kde = gaussian_kde(num_data.T)
                    
                    self.distributions[combination] = {
                        'method': 'kde',
                        'kde': kde,
                        'n_samples': group_size,
                        'original_data': num_data
                    }
                    
                elif self.method == 'bootstrap':
                    self.distributions[combination] = {
                        'method': 'bootstrap',
                        'n_samples': group_size,
                        'original_data': num_data
                    }
                
                successful_fits += 1
                print(f"    ✅ Success")
                
            except Exception as e:
                print(f"    ❌ Failed: {e}")
                print(f"    🔄 Falling back to bootstrap for this group")
                
                self.distributions[combination] = {
                    'method': 'bootstrap',
                    'n_samples': group_size,
                    'original_data': num_data
                }
                successful_fits += 1
        
        self.fitted = True
        print(f"\n✅ FITTING COMPLETE")
        print(f"Successfully fitted {successful_fits} distributions")
        print(f"Total combinations available: {len(self.distributions)}")
        
    def generate_synthetic(self, n_total, preserve_proportions=True, original_df=None):
        """Generate synthetic dataset of specified size"""
        
        if not self.fitted:
            raise ValueError("Model not fitted. Call fit() first.")
            
        print(f"🎲 GENERATING SYNTHETIC DATA")
        print(f"{'='*50}")
        print(f"Target total samples: {n_total}")
        print(f"Preserve proportions: {preserve_proportions}")
        
        synthetic_groups = []
        
        # Calculate sample sizes for each group
        if preserve_proportions and original_df is not None:
            total_original = len(original_df)
            proportions = {}
            for combination in self.distributions.keys():
                group_size = len(original_df.groupby(self.cat_cols).get_group(combination))
                proportions[combination] = group_size / total_original
        else:
            # Equal proportions
            n_groups = len(self.distributions)
            proportions = {combo: 1/n_groups for combo in self.distributions.keys()}
        
        for combination, dist_info in self.distributions.items():
            # Calculate samples for this combination
            n_samples = max(1, int(n_total * proportions[combination]))
            
            print(f"  🔄 Generating {n_samples} samples for {combination}")
            
            if dist_info['method'] == 'multivariate_normal':
                synthetic_num = multivariate_normal.rvs(
                    mean=dist_info['mean'],
                    cov=dist_info['cov'],
                    size=n_samples
                )
                if synthetic_num.ndim == 1:
                    synthetic_num = synthetic_num.reshape(-1, 1)
                    
            elif dist_info['method'] == 'kde':
                synthetic_num = dist_info['kde'].resample(n_samples).T
                
            elif dist_info['method'] == 'bootstrap':
                original_data = dist_info['original_data']
                indices = np.random.choice(len(original_data), size=n_samples, replace=True)
                synthetic_num = original_data[indices]
            
            # Keep synthetic values inside the range seen in the original group.
            # A normal distribution has unbounded tails, so without this step it can
            # produce negative revenue/profit or satisfaction scores above 5.
            original_data = dist_info['original_data']
            synthetic_num = np.clip(synthetic_num, original_data.min(axis=0), original_data.max(axis=0))

            # Create DataFrame for this group
            group_df = pd.DataFrame(synthetic_num, columns=self.num_cols)
            
            # Add categorical columns
            if isinstance(combination, tuple):
                for i, col in enumerate(self.cat_cols):
                    group_df[col] = combination[i]
            else:
                group_df[self.cat_cols[0]] = combination
            
            synthetic_groups.append(group_df)
        
        # Combine and shuffle
        synthetic_df = pd.concat(synthetic_groups, ignore_index=True)
        synthetic_df = synthetic_df.sample(frac=1, random_state=42).reset_index(drop=True)
        
        # Reorder columns to match original
        column_order = self.cat_cols + self.num_cols
        synthetic_df = synthetic_df[column_order]
        
        print(f"\n✅ GENERATION COMPLETE")
        print(f"Generated {len(synthetic_df)} synthetic samples")
        
        return synthetic_df

print("✅ Joint Distribution Synthesizer class defined")

## Step 6: Construct Joint Distributions and Generate Synthetic Data

In [ ]:
# Initialize and fit the synthesizer
synthesizer = JointDistributionSynthesizer(method=GENERATION_METHOD, min_samples=5)
synthesizer.fit(df, cat_cols, num_cols)

# Generate synthetic data
synthetic_df = synthesizer.generate_synthetic(
    n_total=DESIRED_SYNTHETIC_SIZE, 
    preserve_proportions=True, 
    original_df=df
)

print(f"\n📊 DATASET COMPARISON")
print(f"{'='*50}")
print(f"Original dataset: {df.shape}")
print(f"Synthetic dataset: {synthetic_df.shape}")

# Show sample of synthetic data
print(f"\nFirst 5 rows of synthetic data:")
synthetic_df.head()

## Step 7: Statistical Comparison

In [ ]:
def compare_datasets(original_df, synthetic_df, cat_cols, num_cols):
    """Comprehensive statistical comparison between datasets"""
    
    print(f"📈 STATISTICAL COMPARISON")
    print(f"{'='*60}")
    
    # Overall statistics
    print(f"\n1. OVERALL STATISTICS")
    print(f"{'-'*30}")
    
    orig_overall = original_df[num_cols].describe()
    synth_overall = synthetic_df[num_cols].describe()
    
    for col in num_cols:
        orig_mean = orig_overall.loc['mean', col]
        synth_mean = synth_overall.loc['mean', col]
        orig_std = orig_overall.loc['std', col]
        synth_std = synth_overall.loc['std', col]
        
        print(f"{col}:")
        print(f"  Original  - Mean: {orig_mean:.2f}, Std: {orig_std:.2f}")
        print(f"  Synthetic - Mean: {synth_mean:.2f}, Std: {synth_std:.2f}")
        print(f"  Difference - Mean: {abs(orig_mean - synth_mean):.2f}, Std: {abs(orig_std - synth_std):.2f}")
        print()
    
    # Group-wise comparison
    print(f"2. GROUP-WISE COMPARISON")
    print(f"{'-'*30}")
    
    orig_grouped = original_df.groupby(cat_cols)[num_cols].agg(['mean', 'std'])
    synth_grouped = synthetic_df.groupby(cat_cols)[num_cols].agg(['mean', 'std'])
    
    print("Group means comparison:")
    means_comparison = pd.DataFrame()
    for col in num_cols:
        means_comparison[f'{col}_orig'] = orig_grouped[col]['mean']
        means_comparison[f'{col}_synth'] = synth_grouped[col]['mean']
        means_comparison[f'{col}_diff'] = abs(orig_grouped[col]['mean'] - synth_grouped[col]['mean'])
    
    print(means_comparison.round(2))
    
    # Correlation comparison
    print(f"\n3. CORRELATION MATRICES")
    print(f"{'-'*30}")
    
    orig_corr = original_df[num_cols].corr()
    synth_corr = synthetic_df[num_cols].corr()
    
    print("Original correlations:")
    print(orig_corr.round(3))
    print("\nSynthetic correlations:")
    print(synth_corr.round(3))
    print("\nCorrelation differences:")
    print((synth_corr - orig_corr).round(3))
    
    return means_comparison

# Perform comparison
comparison_results = compare_datasets(df, synthetic_df, cat_cols, num_cols)

In [ ]:
comparison_results

## Step 8: Overall Pair Plot Comparison

In [ ]:
# Create overall comparison pair plot
print("Creating overall pair plot comparison...")

# Prepare data for plotting
orig_plot = df.copy()
orig_plot['Dataset'] = 'Original'
if len(cat_cols) == 1:
    orig_plot['cat_combined'] = orig_plot[cat_cols[0]]
else:
    orig_plot['cat_combined'] = orig_plot[cat_cols].apply(
        lambda x: ' + '.join(x.astype(str)), axis=1
    )

synth_plot = synthetic_df.copy()
synth_plot['Dataset'] = 'Synthetic'
if len(cat_cols) == 1:
    synth_plot['cat_combined'] = synth_plot[cat_cols[0]]
else:
    synth_plot['cat_combined'] = synth_plot[cat_cols].apply(
        lambda x: ' + '.join(x.astype(str)), axis=1
    )

# Pair plots (sns.pairplot always creates its own figure, so these appear one after the other)

# Original data pair plot
g1 = sns.pairplot(
    orig_plot[num_cols + ['cat_combined']], 
    hue='cat_combined',
    diag_kind='hist',
    plot_kws={'alpha': 0.7, 's': 20},
    height=2.2
)
g1.fig.suptitle('Original Data - Joint Distributions', y=1.02, fontweight='bold')

# Synthetic data pair plot
g2 = sns.pairplot(
    synth_plot[num_cols + ['cat_combined']], 
    hue='cat_combined',
    diag_kind='hist',
    plot_kws={'alpha': 0.7, 's': 20},
    height=2.2
)
g2.fig.suptitle(f'Synthetic Data - Joint Distributions ({GENERATION_METHOD})', y=1.02, fontweight='bold')

plt.show()

print("✅ Overall comparison complete")

## Step 9: Category-wise Pair Plot Comparisons

**This is the key analysis**: Individual pair plots for each categorical combination showing original vs synthetic data side-by-side.

In [ ]:
def create_category_wise_pairplots(original_df, synthetic_df, cat_cols, num_cols):
    """Create individual pair plots for each categorical combination"""
    
    print(f"🎨 CREATING CATEGORY-WISE PAIR PLOTS")
    print(f"{'='*60}")
    
    # Get all unique combinations
    unique_combinations = original_df.groupby(cat_cols).size().index.tolist()
    
    for i, combination in enumerate(unique_combinations):
        print(f"\n📊 Creating pair plot {i+1}/{len(unique_combinations)}: {combination}")
        
        # Filter data for this combination
        if len(cat_cols) == 1:
            orig_subset = original_df[original_df[cat_cols[0]] == combination]
            synth_subset = synthetic_df[synthetic_df[cat_cols[0]] == combination]
        else:
            # Multiple categorical columns
            orig_mask = True
            synth_mask = True
            for j, col in enumerate(cat_cols):
                orig_mask &= (original_df[col] == combination[j])
                synth_mask &= (synthetic_df[col] == combination[j])
            orig_subset = original_df[orig_mask]
            synth_subset = synthetic_df[synth_mask]
        
        if len(orig_subset) == 0 or len(synth_subset) == 0:
            print(f"  ⚠️  Skipping {combination}: insufficient data")
            continue
        
        print(f"  Original samples: {len(orig_subset)}, Synthetic samples: {len(synth_subset)}")
        
        # Prepare data for plotting
        orig_plot = orig_subset[num_cols].copy()
        orig_plot['Dataset'] = 'Original'
        
        synth_plot = synth_subset[num_cols].copy()
        synth_plot['Dataset'] = 'Synthetic'
        
        # Combine for overlay plotting
        combined_plot = pd.concat([orig_plot, synth_plot], ignore_index=True)
        
        # Create pair plot
        g = sns.pairplot(
            combined_plot,
            hue='Dataset',
            diag_kind='hist',
            plot_kws={'alpha': 0.6, 's': 30},
            diag_kws={'alpha': 0.7},
            height=2.5,
            palette=['blue', 'red']
        )
        
        # Customize the plot
        combination_str = str(combination) if len(cat_cols) == 1 else ' + '.join(map(str, combination))
        g.fig.suptitle(
            f'Comparison for {combination_str}\nOriginal vs Synthetic Data Joint Distributions',
            y=1.02, fontsize=14, fontweight='bold'
        )
        
        # Improve legend
        g._legend.set_title('Dataset Type')
        g._legend.set_bbox_to_anchor((1.02, 0.8))
        
        plt.tight_layout()
        plt.show()
        
        # Print statistics for this combination
        print(f"  📈 Statistics for {combination_str}:")
        
        for col in num_cols:
            orig_mean = orig_subset[col].mean()
            synth_mean = synth_subset[col].mean()
            orig_std = orig_subset[col].std()
            synth_std = synth_subset[col].std()
            
            print(f"    {col}: Orig(μ={orig_mean:.2f}, σ={orig_std:.2f}) vs Synth(μ={synth_mean:.2f}, σ={synth_std:.2f})")
        
        # Correlation comparison for this group
        if len(num_cols) > 1:
            orig_corr = orig_subset[num_cols].corr()
            synth_corr = synth_subset[num_cols].corr()
            
            print(f"    Correlation preservation quality:")
            corr_diff = abs(orig_corr - synth_corr).values
            avg_corr_diff = np.mean(corr_diff[np.triu_indices_from(corr_diff, k=1)])
            print(f"    Average correlation difference: {avg_corr_diff:.3f}")
        
        print(f"  ✅ Completed pair plot for {combination_str}")
    
    print(f"\n🎊 ALL CATEGORY-WISE PAIR PLOTS COMPLETED!")
    print(f"Review the plots above to see how well joint distributions were preserved for each category combination.")

# Create the category-wise pair plots
create_category_wise_pairplots(df, synthetic_df, cat_cols, num_cols)

## Step 10: Save Results and Summary

In [ ]:
# Save datasets
df.to_csv('original_dataset.csv', index=False)
synthetic_df.to_csv(f'synthetic_dataset_{GENERATION_METHOD}.csv', index=False)

print(f"💾 SAVED FILES:")
print(f"- original_dataset.csv")
print(f"- synthetic_dataset_{GENERATION_METHOD}.csv")

# Save detailed analysis
analysis_summary = {
    'original_shape': df.shape,
    'synthetic_shape': synthetic_df.shape,
    'categorical_columns': cat_cols,
    'numerical_columns': num_cols,
    'generation_method': GENERATION_METHOD,
    'unique_combinations': len(df.groupby(cat_cols)),
    'distributions_fitted': len(synthesizer.distributions)
}

with open(f'analysis_summary_{GENERATION_METHOD}.txt', 'w') as f:
    f.write("Joint Distribution Analysis Summary\n")
    f.write("=" * 40 + "\n\n")
    for key, value in analysis_summary.items():
        f.write(f"{key}: {value}\n")

print(f"- analysis_summary_{GENERATION_METHOD}.txt")

# Final comprehensive summary
print(f"\n{'='*70}")
print(f"🎯 JOINT DISTRIBUTION ANALYSIS - FINAL SUMMARY")
print(f"{'='*70}")

print(f"📊 DATA OVERVIEW:")
print(f"   Original dataset: {df.shape[0]:,} samples, {df.shape[1]} features")
print(f"   Synthetic dataset: {synthetic_df.shape[0]:,} samples, {synthetic_df.shape[1]} features")
print(f"   Categorical columns: {len(cat_cols)} ({', '.join(cat_cols)})")
print(f"   Numerical columns: {len(num_cols)} ({', '.join(num_cols)})")

print(f"\n🔧 METHODOLOGY:")
print(f"   Generation method: {GENERATION_METHOD}")
print(f"   Unique categorical combinations: {len(df.groupby(cat_cols))}")
print(f"   Joint distributions fitted: {len(synthesizer.distributions)}")
print(f"   Minimum samples per group: 5")

print(f"\n📈 VALIDATION PERFORMED:")
print(f"   ✅ Overall statistical comparison (means, std devs)")
print(f"   ✅ Group-wise statistical comparison")
print(f"   ✅ Correlation matrix comparison")
print(f"   ✅ Overall pair plots (original vs synthetic)")
print(f"   ✅ Category-wise pair plots (each combination)")

print(f"\n🎊 ANALYSIS COMPLETE!")
print(f"\n💡 KEY INSIGHTS:")
print(f"   • Joint distributions were constructed for each categorical combination")
print(f"   • Synthetic data preserves statistical relationships within each group")
print(f"   • Pair plots show visual validation of distribution preservation")
print(f"   • Method '{GENERATION_METHOD}' was used for generation")

print(f"\n🔄 TO USE WITH DIFFERENT SETTINGS:")
print(f"   • Change DESIRED_SYNTHETIC_SIZE at the top")
print(f"   • Try different GENERATION_METHOD: 'multivariate_normal', 'kde', 'bootstrap'")
print(f"   • Update CSV_FILE_PATH to your data file")

print(f"\n📝 REMEMBER:")
print(f"   • Ensure your CSV has 'cat_' and 'num_' prefixed columns")
print(f"   • Review category-wise pair plots for quality assessment")
print(f"   • Multivariate normal works best for Gaussian-like data")
print(f"   • KDE is more flexible for non-Gaussian distributions")
print(f"   • Bootstrap preserves exact empirical distributions")

print(f"{'='*70}")